In [2]:
import mlflow
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from data_validation import validate_training_data

# 1. Load the data and check that every sample has a label.
X, y = load_digits(return_X_y=True)
validate_training_data(X, y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=67, stratify=y
)

# 2. Store experiment records in a local database.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("digits-classification")

# 3. Train one model and record its result.
max_depth = 8
with mlflow.start_run(run_name=f"tree-depth-{max_depth}") as run:
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=67)
    model.fit(X_train, y_train)
    accuracy = model.score(X_val, y_val)

    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("validation_accuracy", accuracy)

    print(f"Run ID: {run.info.run_id}")
    print(f"Validation accuracy: {accuracy:.2%}")

Run ID: 9426fc38ac584fa5ba63bd54996c3bd9
Validation accuracy: 87.22%


In [4]:
import mlflow.sklearn

with mlflow.start_run(run_id=run.info.run_id):
    saved_model = mlflow.sklearn.log_model(
        sk_model=model,
        name="decision-tree",
        input_example=X_val[:1],
        registered_model_name="digits-decision-tree",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

print("Registered model: digits-decision-tree")
print("Version:", saved_model.registered_model_version)

Registered model: digits-decision-tree
Version: 1


Successfully registered model 'digits-decision-tree'.
Created version '1' of model 'digits-decision-tree'.


In [5]:
loaded_model = mlflow.sklearn.load_model("models:/digits-decision-tree/1")

original_predictions = model.predict(X_val)
loaded_predictions = loaded_model.predict(X_val)

print("Same predictions:", (original_predictions == loaded_predictions).all())
print(f"Loaded model accuracy: {loaded_model.score(X_val, y_val):.2%}")

Same predictions: True
Loaded model accuracy: 87.22%
